# Training Tracker

Notebook for monitoring and evaluating the 2-bit input compression pipeline. Covers:
- Loss curves for **Part 1** (soft quantize layer) and **Part 2** (2-bit optimized) training
- Threshold optimization curves learned during **Part 1**
- Residual plots for model evaluation on the test dataset

## Setup

Update `sys.path.insert(0, ...)` in the cell below to point to your local `two_bit_optimization_helpers` directory.

In [ ]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import sys

sys.path.insert(0, "./two_bit_optimization_helpers")  # update to your local path

# parse checkpoint filenames and extract train/val losses or thresholds per epoch
from train import create_model, get_best_thresholds, cleanup_models_and_generators, get_all_losses, get_all_thresholds

In [ ]:
# === Parameters — only these two values need to change between runs ===
# (must match the values used in two_bit_optimization.ipynb)
model_type = 'Mlp_Slim'   # e.g. Mlp_Slim, Conv2D_Full, QConv2D_Slim …
use_roi    = False         # True → dataset_2su_roi test set; False → dataset_2sc

# Path to training dataset — needed to locate TFRecords metadata.json
# (same value as dataset_3src_dir in two_bit_optimization.ipynb)
dataset_3src_dir = '/nas/work/research/smartpix-box/pixelAV_datasets/shuffled/largerWindowPreliminary/dataset_3src_16x16_50x12P5_centeredIncidence_parquets'

# --- derived automatically — do not edit below ---
import glob, json

_weights_base = 'smart-pixels/weights/dataset_3src_16x16_50x12P5_centeredIncidence_weights'

# infer fingerprint from the Part 2 checkpoint directory name
# old hardcoded value: '392456de'
_dirs = glob.glob(f'{_weights_base}/weights-2t-{model_type}-2bit_optimized-*-checkpoints')
assert len(_dirs) == 1, f"Expected 1 Part 2 checkpoint dir for {model_type}, found: {_dirs}"
fingerprint = _dirs[0].split(f'{model_type}-2bit_optimized-')[1].split('-checkpoints')[0]
print(f"fingerprint : {fingerprint}")

# read labels_scale from TFRecords metadata.json
# old hardcoded values: [122.89689703635774, 30.903849401109394, 1.917222249583349]
_slim_suffix = '_slim' if 'Slim' in model_type else ''
_meta = f'{dataset_3src_dir}/TFR_files/2t/TFR_test_contained{_slim_suffix}/metadata.json'
with open(_meta) as _f:
    labels_scale = json.load(_f)['labels_scale']
print(f"labels_scale: {labels_scale}")

## Loss Curve

Plots training and validation loss across epochs for a given checkpoint directory (Part 1 or Part 2).

**Arguments to change:**
- `model_checkpoints`
  -  path to the checkpoint directory produced by `train()`
  -  e.g. `weights-2t-Mlp_Slim-2bit_optimized-<fingerprint>-checkpoints/`

In [ ]:
# old: model_type = 'Mlp_Slim'  — now set in the Parameters cell above

# old: model_checkpoints = f'smart-pixels/weights/dataset_3src_16x16_50x12P5_centeredIncidence_weights/weights-2t-{model_type}-2bit_optimized-392456de-checkpoints/'
model_checkpoints = f'{_weights_base}/weights-2t-{model_type}-2bit_optimized-{fingerprint}-checkpoints/'

# parse loss values from checkpoint filenames (weights.NN-tX.XXX-vY.YYY.hdf5)
train_losses, validation_losses = get_all_losses(model_checkpoints)
epochs = np.arange(1, len(train_losses) + 1)

fig, ax = plt.subplots()
ax.scatter(epochs, train_losses,      s=2, c='red',  label='training loss')
ax.scatter(epochs, validation_losses, s=2, c='blue', label='validation loss')
ax.set_xlabel('Epoch')
ax.set_ylabel('Loss')
ax.legend()
fig.suptitle('Loss vs Epoch')
fig.tight_layout()
# fig.savefig('loss_curve.png', dpi=300)
fig.show()

## Threshold Optimization Curve (Part 1)

Plots how the 3 learned charge thresholds evolved across epochs during Part 1 (soft quantize layer) training.

**Arguments to change:**
- `input_dir` — path to the Part 1 checkpoint directory (`weights-2t-<model>-soft_quantize_layer-<fingerprint>-checkpoints/`)
- `model_type` — model type used in training (e.g. `Mlp_Slim`, `Conv2D_Full`)

**Optional:**
- `threshold_offset` — default `80.0`; match the value used during training
- `initial_levels` — default `[0.0, 1.0, 2.0, 3.0]`; match the value used during training
- `timeslices` — default `2`

In [ ]:
# load the soft_quantize_layer model at each checkpoint and extract its 3 learned thresholds
# old: input_dir=f'smart-pixels/weights/dataset_3src_16x16_50x12P5_centeredIncidence_weights/weights-2t-{model_type}-soft_quantize_layer-392456de-checkpoints/'
thresholds_1, thresholds_2, thresholds_3 = get_all_thresholds(
    input_dir=f'{_weights_base}/weights-2t-{model_type}-soft_quantize_layer-{fingerprint}-checkpoints/',
    model_type=model_type,
    threshold_offset=80.0,
    initial_levels=np.array([0.0, 1.0, 2.0, 3.0]),
    timeslices=2,
)

epochs = np.arange(1, len(thresholds_1) + 1)

# one subplot per threshold; y-axis shared so epoch progression is easy to compare
fig, axes = plt.subplots(1, 3, figsize=(12, 4), sharey=True)
for ax, thresholds, color, label in zip(
    axes,
    [thresholds_1, thresholds_2, thresholds_3],
    ['red', 'orange', 'green'],
    ['threshold 1', 'threshold 2', 'threshold 3'],
):
    ax.scatter(thresholds, epochs, s=2, c=color, label=label)
    ax.set_xlabel('Charge (e)')
    ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=False, nbins=6))
    ax.legend()

axes[0].set_ylabel('Epoch')
fig.suptitle('Charge Threshold Progression')
fig.tight_layout()
# fig.savefig('threshold_progression.png', dpi=300)
fig.show()

## Best Thresholds (Part 1 → Part 2)

Extracts and prints the charge thresholds from the best Part 1 checkpoint. These values are passed as `initial_thresholds` when digitizing inputs for Part 2 training.

**Arguments to change:**
- `checkpoints` — path to the Part 1 checkpoint directory
- `model_type` — model type used in training (e.g. `Mlp_Slim`, `Conv2D_Full`)
- `initial_thresholds` — starting thresholds used during Part 1 training

**Optional:**
- `threshold_offset` — default `80.0`; match the value used during training
- `initial_levels` — default `[0.0, 1.0, 2.0, 3.0]`; match the value used during training
- `timeslices` — default `2`

**Outputs:** `thresholds`, `levels` — printed by `get_best_thresholds`

In [ ]:
# find the best checkpoint (lowest val_loss), load it, and print its thresholds and levels
# use these threshold values as initial_thresholds in Part 2 training
# old: checkpoints=f'smart-pixels/weights/dataset_3src_16x16_50x12P5_centeredIncidence_weights/weights-2t-{model_type}-soft_quantize_layer-392456de-checkpoints'
thresholds, levels = get_best_thresholds(
    checkpoints=f'{_weights_base}/weights-2t-{model_type}-soft_quantize_layer-{fingerprint}-checkpoints',
    model_type=model_type,
    initial_thresholds=[247.8, 668.4, 1662.9],
    threshold_offset=80.0,
    initial_levels=np.array([0.0, 1.0, 2.0, 3.0]),
    timeslices=2,
)
# Outputs: thresholds (3 values in e), levels (4 values)

## Residual Plots

Plots true-minus-predicted residuals as a function of true value, binned across the output space. For Max/Full models the predicted uncertainty band (`sigma`) is also overlaid.

`labels_scale` rescales the normalized model outputs back to physical units — load it from the `metadata.json` in the test TFRecords directory.

In [ ]:
def residual_plot(ax, thisdf, var1, var2, name, color, label=None, scaling=1.0, alpha=0.2, slim=False, edgecolor=None, hatch=None):
    """Scatter residuals (true - predicted) binned by true value. Overlays uncertainty band for non-Slim models."""
    nbins = 50

    # rescale from normalized model output to physical units
    var1_scaled = thisdf[var1] * scaling
    var2_scaled = thisdf[var2] * scaling
    residual_scaled = var1_scaled - var2_scaled

    xmin, xmax = np.min(var1_scaled), np.max(var1_scaled)
    step = (xmax - xmin) / nbins

    # binned scatter: each point is the mean residual in that bin
    x = sns.regplot(x=var1_scaled, y=residual_scaled, x_bins=np.linspace(xmin, xmax, nbins),
                    fit_reg=None, marker='.', ax=ax, color=color, label=label)
    ax.set_xlabel('True ' + name)
    ax.set_ylabel('True - predicted ' + name)

    thisdf['residual' + var2] = residual_scaled

    if not slim:
        # overlay ±1σ uncertainty band predicted by the model (not available for Slim)
        means, upbar, downbar = [], [], []
        for i in range(nbins):
            mask = (var1_scaled > xmin + i * step) & (var1_scaled < xmin + (i + 1) * step)
            means.append(np.mean(thisdf['residual' + var2][mask]))
            sigma_mean = np.mean(thisdf['sigma' + var2][mask] * scaling)
            upbar.append(means[-1] + sigma_mean)
            downbar.append(means[-1] - sigma_mean)
        ax.fill_between(np.linspace(xmin, xmax, nbins), upbar, downbar,
                        alpha=alpha, color=color, edgecolor=edgecolor, hatch=hatch)


def inverse_cot(cota):
    # arctan(1/x) with branch correction to keep angles in [0, π]
    a = np.arctan(1.0 / cota)
    a[np.where(a < 0)] += np.pi
    return a


def residual_plot_deg(ax, thisdf, var1, var2, name, color, label=None, scaling=1.0, alpha=0.2, slim=False, edgecolor=None, hatch=None):
    """Like residual_plot but converts cotangent values to degrees before plotting."""
    if 'cot' not in var1:
        residual_plot(ax, thisdf, var1, var2, name, scaling=scaling)
        return

    # convert cot → angle in degrees for both prediction and truth
    thisdf['angle']     = inverse_cot(thisdf[var2].values * scaling) * 180 / np.pi
    thisdf['angletrue'] = inverse_cot(thisdf[var1].values * scaling) * 180 / np.pi

    if not slim:
        # propagate ±σ through the cot→deg conversion
        thisdf['angleup']   = abs(inverse_cot((thisdf[var2].values + thisdf['sigma' + var2].values) * scaling) * 180 / np.pi - thisdf['angle'])
        thisdf['angledown'] = abs(inverse_cot((thisdf[var2].values - thisdf['sigma' + var2].values) * scaling) * 180 / np.pi - thisdf['angle'])

    var1, var2 = 'angletrue', 'angle'
    nbins = 50
    xmin, xmax = np.min(thisdf[var1]), np.max(thisdf[var1])
    step = (xmax - xmin) / nbins

    # binned scatter in degree space
    x = sns.regplot(x=thisdf[var1], y=(thisdf[var1] - thisdf[var2]), x_bins=np.linspace(xmin, xmax, nbins),
                    fit_reg=None, marker='.', ax=ax, color=color, label=label)
    ax.set_xlabel('True ' + name)
    ax.set_ylabel('True - predicted ' + name)

    thisdf['residual' + var2] = thisdf[var1] - thisdf[var2]

    if not slim:
        means, upbar, downbar = [], [], []
        for i in range(nbins):
            mask = (thisdf[var1] > xmin + i * step) & (thisdf[var1] < xmin + (i + 1) * step)
            means.append(np.mean(thisdf['residual' + var2][mask]))
            upbar.append(means[-1] + np.mean(thisdf['angleup'][mask]))
            downbar.append(means[-1] - np.mean(thisdf['angledown'][mask]))
        ax.fill_between(np.linspace(xmin, xmax, nbins), upbar, downbar,
                        alpha=alpha, color=color, edgecolor=edgecolor, hatch=hatch)

### Max / Full Models

**Arguments to change:**
- `performance_parquet` — path to the output parquet from `save_performance_parquet()` (contains predictions and truth for the test set)
- `labels_scale` — 4-element list `[scale_x, scale_y, scale_cotA, scale_cotB]` from `metadata.json`

### Slim Models

**Arguments to change:**
- `performance_parquet` — path to the output parquet from `save_performance_parquet()`
- `labels_scale` — 3-element list `[scale_x, scale_y, scale_cotB]` from `metadata.json`

In [ ]:
# old: use_roi = False  — now set in the Parameters cell above

dataset_label = 'ROI' if use_roi else '2sc'
_base = 'smart-pixels-ml/processed_parquets/dataset_3src_16x16_50x12P5_centeredIncidence'
#_test = 'test_dataset_2su_-9_9_48x192_50x12P5_roi' if use_roi else 'test_dataset_2sc_16x16_50x12P5_centeredIncidence'
_test = 'test_dataset_2s_48x192_50x12P5_roi' if use_roi else 'test_dataset_2sc_16x16_50x12P5_centeredIncidence'

# paths to performance parquets — fingerprint derived automatically from Parameters cell
# old: …-392456de-vars.parquet
performance_parquet_mlp_slim    = f'{_base}/{_test}/2bit_optimized/2t-Mlp_Slim-2bit_optimized-{fingerprint}-vars.parquet'
performance_parquet_conv2d_slim = f'{_base}/{_test}/2bit_optimized/2t-Conv2D_Slim-2bit_optimized-{fingerprint}-vars.parquet'
performance_parquet_conv1d_slim = f'{_base}/{_test}/2bit_optimized/2t-Conv1D_Slim-2bit_optimized-{fingerprint}-vars.parquet'

# labels_scale read from metadata.json in Parameters cell
# old hardcoded values: [122.89689703635774, 30.903849401109394, 1.917222249583349]

models = [
    ('Mlp_Slim',    performance_parquet_mlp_slim,    'red'),
    #('Conv1D_Slim', performance_parquet_conv1d_slim, 'blue'),
    #('Conv2D_Slim', performance_parquet_conv2d_slim, 'green'),
]

fig, ax = plt.subplots(1, 3, figsize=(20, 6))

for model_name, parquet_path, color in models:
    df = pd.read_parquet(parquet_path)
    # Slim models do not predict uncertainties (slim=True disables the sigma band)
    residual_plot(ax[0], df, 'xtrue', 'x',           name=r'$x$ $[\mu m]$',   color=color, scaling=labels_scale[0], slim=True, label=model_name)
    residual_plot(ax[1], df, 'ytrue', 'y',           name=r'$y$ $[\mu m]$',   color=color, scaling=labels_scale[1], slim=True, label=model_name)
    residual_plot_deg(ax[2], df, 'cotBtrue', 'cotB', name=r'$\beta$ $[deg]$', color=color, scaling=labels_scale[2], slim=True, label=model_name)

for a, title in zip(ax, [r'$x$', r'$y$', r'$\beta$']):
    a.axhline(alpha=0.4, ls='dashed')
    a.set_title(f'{title} residuals — {dataset_label}')
    a.legend()

fig.tight_layout(pad=1.0)
# fig.savefig(f'residuals_slim_{dataset_label}.png', bbox_inches='tight', dpi=300)
fig.show()